# 🟧 RT-DETR — Drowsy Driver 6-class (Colab T4)

**Transformer #2** — RT-DETR (Baidu), NMS-free, nằm trong `ultralytics` nên train y như YOLO.
Dùng để **ensemble** với YOLO26 (CNN) và RF-DETR (transformer DINOv2).

| | RT-DETR | RF-DETR | YOLO26 |
|---|---|---|---|
| Backbone | ResNet/HGNet | DINOv2 | CNN (C3k2) |
| Framework | ultralytics | rfdetr | ultralytics |
| Format data | yolo (txt) | coco (json) | yolo (txt) |
| NMS | không | không | không |

`Runtime → Change runtime type → T4 GPU → Run all`

In [ ]:
# 1 — GPU + cài đặt
!nvidia-smi -L
%pip install -q -U "ultralytics>=8.4.0" supervision roboflow
import os, glob, shutil, json
from pathlib import Path
import ultralytics; ultralytics.checks()
HOME = Path('/content')

try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY'); assert ROBOFLOW_API_KEY
except Exception:
    ROBOFLOW_API_KEY = 'qI3lEKlNpIZpNENdk3MH'    # ← key của bạn

In [ ]:
# 2 — Download dataset (format yolov11 — RT-DETR dùng định dạng YOLO như YOLO26)
import yaml
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

dataset = None
for proj in ['datio_yolo', 'driver-yawn', 'driver-yawn-wh6wj']:
    try:
        dataset = rf.workspace('nguyen-tuan-dat').project(proj).version(1).download('yolov11')
        print('✅  Dùng project:', proj); break
    except Exception:
        print('  ⏭️ ', proj, 'không tải được...')

# Vá data.yaml từ cấu trúc thư mục
loc = Path(dataset.location)
old = {}
for yp in list(loc.glob('*.yaml')):
    with open(yp) as f: t = yaml.safe_load(f) or {}
    if 'names' in t: old = t; break
names = old.get('names')
if isinstance(names, dict): names = [names[k] for k in sorted(names)]
if not names: names = ['close_eyeL','close_eyeR','no_yawn','open_eyeL','open_eyeR','yawn']
def imgdir(*c):
    for x in c:
        d = loc/x
        if d.exists() and any(d.iterdir()): return str(d.resolve())
    return None
cfg = {'train': imgdir('train/images','train'),
       'val':   imgdir('valid/images','valid','val') or imgdir('test/images','test'),
       'nc': len(names), 'names': names}
_t = imgdir('test/images','test')
if _t: cfg['test'] = _t
DATA_YAML = f'{dataset.location}/data.yaml'
with open(DATA_YAML,'w') as f: yaml.dump(cfg, f, sort_keys=False)
print('✅  data.yaml:', cfg)

In [ ]:
# 3 — TRAIN RT-DETR (transformer nặng RAM hơn → batch 8 trên T4)
from ultralytics import RTDETR

MODEL   = 'rtdetr-l'      # rtdetr-l (32M) hoặc rtdetr-x (67M, cần GPU mạnh hơn)
model = RTDETR(f'{MODEL}.pt')
model.train(
    data     = DATA_YAML,
    epochs   = 60,
    imgsz    = 640,         # RT-DETR bắt buộc 640
    batch    = 8,           # transformer nặng → 8 cho T4
    patience = 20,
    plots    = True,
    name     = 'drowsy_rtdetr',
    project  = str(HOME/'runs'/'detect'),
    cos_lr   = True, lr0 = 0.0001, weight_decay = 0.0001,
    device   = 0,
)

In [ ]:
# 4 — Validate + mAP per-class
TRAIN_DIR = max(glob.glob(f'{HOME}/runs/detect/drowsy_rtdetr*'), key=os.path.getmtime)
BEST = f'{TRAIN_DIR}/weights/best.pt'
m = RTDETR(BEST)
metrics = m.val(data=DATA_YAML, verbose=True)
print(f'\n  RT-DETR  mAP50 = {metrics.box.map50*100:.2f}%  |  mAP50-95 = {metrics.box.map*100:.2f}%')
for i,c in enumerate(names):
    try: print(f'    {c:<12}: {metrics.box.maps[i]*100:.2f}%')
    except: pass

In [ ]:
# 5 — Plots + predict demo
from IPython.display import Image as IPyImage, display
import matplotlib.pyplot as plt, cv2
for f in ['results.png','confusion_matrix.png']:
    p = f'{TRAIN_DIR}/{f}'
    if os.path.exists(p): display(IPyImage(filename=p, width=720))

test_src = f'{dataset.location}/test/images'
m.predict(source=test_src, conf=0.35, save=True, project=str(HOME/'runs'/'detect'), name='rtdetr_pred', verbose=False)
preds = sorted(glob.glob(f'{HOME}/runs/detect/rtdetr_pred*/*.jpg'))[:6]
if preds:
    fig, ax = plt.subplots(2,3, figsize=(15,8))
    for a,p in zip(ax.flat, preds):
        a.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)); a.axis('off')
    plt.suptitle('RT-DETR — 6-class', fontweight='bold'); plt.tight_layout(); plt.show()

In [ ]:
# 6 — Lưu Drive (cho file ensemble)
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/DrowsyDriver_Results'); OUT.mkdir(parents=True, exist_ok=True)
shutil.copy(BEST, OUT/'rtdetr_best.pt')
(OUT/'summary_rtdetr.json').write_text(json.dumps(
    {'model':MODEL,'classes':names,
     'map50':round(float(metrics.box.map50),4),
     'map50_95':round(float(metrics.box.map),4)}, indent=2))
print('✅  Lưu Drive: rtdetr_best.pt + summary_rtdetr.json')